[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weilinear/rsi_labs/blob/main/trajectory_improvement.ipynb)
# Does a bill agent actually need a calculator?

The earlier toy wrote a tool after one correct manual answer. Here, a **fixed
improver reads real development trajectories** and chooses a task-instruction
change, a new arithmetic tool, or no change. We compare its proposal with an
instruction-only proposal and the original agent on unseen bills.

This is agent improvement. **The improver never edits itself.** The model,
objective, evaluator, dataset, tool contract, and execution harness stay fixed.

**Recorded result:** the original agent got 96/96 attempts correct across all
48 distinct bills. Both proposal modes chose `no_change`; all 192 additional
unchanged-policy attempts were also correct. No tool was written or used, and no
accuracy improvement was demonstrated. This is not a manual-versus-tool ablation.
The full run used 290 calls and 220,109 reported tokens (excluding four preliminary
smoke-test calls). Saved outputs and an expandable record are below.

## Setup
Add `GOOGLE_API_KEY` to Colab Secrets and enable notebook access. Live runs send
synthetic bills, responses, and generated code to Google and consume API quota.
No companion library is needed. Local Jupyter users can skip the Colab-secret
cell if their kernel already has `GOOGLE_API_KEY`.

In [ ]:
%pip -q install google-genai==2.28.0

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

## 1. Bills, evaluator, and a small tool boundary
We generate 48 bills: ordinary amounts, fractional rates, and rounding edges.
Each group contributes eight development and eight held-out bills. Each bill gets
two independent attempts at temperature zero; repeats reveal run variability,
not independent new tasks. The half-up rule and tip basis are explicit.

The objective is exact per-person cents, with one model call per bill. Tokens,
request latency, and tool usage are reported separately. A tool request returns
its executed result directly as the final answer; there is no second model call.
This simple JSON tool protocol is part of the fixed harness.

The following helper accepts only a small arithmetic Python grammar. It is a toy
validator, not a general code sandbox. The oracle is never exposed to the task
agent, and the improver never sees held-out bills or answers.

In [1]:
import os, ast, re, json, random, time, statistics
from decimal import Decimal, ROUND_HALF_UP
from concurrent.futures import ThreadPoolExecutor
from google import genai
from google.genai import types

MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
REPEATS, WORKERS = 2, 6
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"], vertexai=False,
    http_options=types.HttpOptions(timeout=45000,
        retry_options=types.HttpRetryOptions(attempts=1)))
usage = []

def ask(prompt, schema):
    started = time.monotonic()
    response = client.models.generate_content(model=MODEL, contents=prompt,
        config={"temperature": 0, "max_output_tokens": 8192,
            "response_mime_type": "application/json", "response_json_schema": schema,
            "automatic_function_calling": {"disable": True}})
    elapsed = time.monotonic() - started
    metadata = {"tokens": response.usage_metadata.total_token_count or 0,
                "seconds": round(elapsed, 3)}
    usage.append(metadata)
    if not response.candidates or response.candidates[0].finish_reason != "STOP":
        raise RuntimeError("Incomplete response")
    return json.loads(response.text), metadata

def expected(task):
    s, tax, tip = (Decimal(task[k]) for k in ("subtotal", "tax", "tip"))
    tax_amount = s * tax / 100
    tip_basis = s if task["tip_on"] == "subtotal" else s + tax_amount
    return ((s + tax_amount + tip_basis * tip / 100) / task["diners"]).quantize(
        Decimal("0.01"), rounding=ROUND_HALF_UP)

def make_bills():
    rng = random.Random(20261005)
    development, held_out = [], []
    for group in ("ordinary", "fractional", "rounding"):
        for i in range(16):
            s = str(Decimal(rng.randrange(501, 500000)) / 100)
            tax, tip = rng.choice(["0", "5", "8", "10"]), rng.choice(["10", "15", "20"])
            diners = rng.randrange(2, 17)
            if group == "fractional":
                tax, tip = rng.choice(["7.25", "8.875", "9.125"]), rng.choice(["17.5", "18.25", "19.625"])
            if group == "rounding":
                # Includes exact half-cent ties, plus non-terminating divisions.
                s = str(Decimal(2 * rng.randrange(500, 10000) + 1) / 100)
                tax, tip, diners = "0", "0", (2 if i % 2 == 0 else 3)
            task = {"id": f"{group}-{i:02}", "group": group, "subtotal": s,
                    "tax": tax, "tip": tip, "diners": diners,
                    "tip_on": "subtotal" if i % 2 == 0 else "total"}
            (development if i < 8 else held_out).append(task)
    return development, held_out

development, held_out = make_bills()
assert not ({t["id"] for t in development} & {t["id"] for t in held_out})
assert expected({"subtotal":"10.01", "tax":"0", "tip":"0", "diners":2,
                 "tip_on":"subtotal"}) == Decimal("5.01")
print(MODEL, "|", len(development), "development bills |", len(held_out), "held-out bills")

def run_tool(source, name, arguments, extra=None):
    tree = ast.parse(source)
    allowed = (ast.Module, ast.FunctionDef, ast.arguments, ast.arg, ast.Assign,
        ast.Return, ast.Expr, ast.Name, ast.Load, ast.Store, ast.Constant,
        ast.BinOp, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.IfExp, ast.Compare,
        ast.Eq, ast.Call, ast.Attribute, ast.keyword)
    nodes = list(ast.walk(tree))
    if len(source) > 2500 or len(nodes) > 220 or len(tree.body) != 1:
        raise ValueError("Tool exceeds the small arithmetic grammar")
    fn = tree.body[0]
    if not isinstance(fn, ast.FunctionDef) or fn.name != name or fn.decorator_list:
        raise ValueError("Expected exactly the requested function")
    if fn.args.defaults or fn.args.kw_defaults or fn.args.vararg or fn.args.kwarg:
        raise ValueError("No defaults or variadic arguments")
    approved = {"Decimal", "int", "str"} | set(extra or {})
    for node in nodes:
        if not isinstance(node, allowed):
            raise ValueError("Disallowed Python syntax: " + type(node).__name__)
        if isinstance(node, (ast.Name, ast.arg)) and "__" in getattr(node, "id", getattr(node, "arg", "")):
            raise ValueError("Disallowed name")
        if isinstance(node, ast.Attribute) and node.attr != "quantize":
            raise ValueError("Only Decimal.quantize is allowed")
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id not in approved:
                raise ValueError("Disallowed function call")
            if not isinstance(node.func, (ast.Name, ast.Attribute)):
                raise ValueError("Disallowed call target")
        if isinstance(node, ast.Constant) and len(str(node.value)) > 100:
            raise ValueError("Constant too large")
    scope = {"__builtins__": {}, "Decimal": Decimal, "int": int, "str": str,
             "ROUND_HALF_UP": ROUND_HALF_UP, **(extra or {})}
    exec(compile(tree, "generated_arithmetic_tool", "exec"), scope)
    return scope[name](*arguments), scope[name]

gemini-3.8-flash | 24 development bills | 24 held-out bills


## 2. Collect baseline trajectories
Every record includes the request, response, displayed calculation summary,
reference answer, correctness, and resource usage. Errors count as unsuccessful
attempts; API failures are flagged separately and never described as arithmetic
failures. Responses are not hidden internal reasoning.

In [2]:
PARAMS = {"subtotal": {"type":"string"}, "tax": {"type":"string", "description":"Bare percent number, without a percent sign"},
          "tip": {"type":"string", "description":"Bare percent number, without a percent sign"}, "diners": {"type":"integer"},
          "tip_on": {"type":"string", "enum":["subtotal", "total"]}}
TASK_SCHEMA = {"type":"object", "properties":{
    "action":{"type":"string", "enum":["answer", "tool"]},
    "amount":{"type":"string", "description":"Dollar amount as a decimal numeric string"}, "calculation":{"type":"string"},
    "arguments":{"type":"object", "properties":PARAMS,
                 "required":list(PARAMS), "additionalProperties":False}},
    "required":["action", "amount", "calculation"], "additionalProperties":False}
BASELINE = {"instruction":"Calculate the bill accurately and answer the user's request.",
            "tool_code":"", "change":"no_change"}

def request(task):
    basis = "subtotal before tax" if task["tip_on"] == "subtotal" else "total after tax"
    return (f"Bill subtotal USD {task['subtotal']}; tax {task['tax']}%; "
        f"tip {task['tip']}% on the {basis}; split equally among {task['diners']} diners. "
        "Keep full precision in intermediate amounts; round only the final per-person "
        "amount to cents using ROUND_HALF_UP (ties round up).")

def trajectory(job):
    artifact, task, repeat = job
    record = {"id":task["id"], "group":task["group"], "repeat":repeat,
              "request":request(task), "reference":str(expected(task))}
    tool = artifact.get("tool")
    contract = ("You may use split_bill(subtotal, tax, tip, diners, tip_on), with dollar "
        "and percentage strings and an integer diner count. Return action='tool' "
        "with arguments, or action='answer' with amount. The host uses the tool result "
        "directly as the final answer." if tool else
        "No tools exist. Return action='answer' and compute the dollar amount yourself.")
    try:
        answer, resources = ask(artifact["instruction"] + "\n" + contract +
            "\nGive a short calculation summary (at most 40 words), not private reasoning.\n" +
            record["request"], TASK_SCHEMA)
        record.update(response=answer, **resources)
    except Exception as error:
        record.update(correct=False, api_error=type(error).__name__)
        return record
    try:
        if answer["action"] == "tool":
            if tool is None:
                raise ValueError("No tool is available")
            args = answer["arguments"]
            amount = tool(*(args[k] for k in PARAMS))
            record["tool_result"] = str(amount)
        else:
            text = answer["amount"].strip()
            # Accept ordinary currency rendering without hiding arithmetic errors.
            if not re.fullmatch(r"(?:USD\s*|\$)?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?", text):
                raise ValueError("Unparseable amount")
            amount = Decimal(text.removeprefix("USD").strip().removeprefix("$").replace(",", ""))
        record.update(amount=str(amount), correct=(amount == expected(task)))
    except Exception as error:
        record.update(correct=False, execution_error=type(error).__name__)
    return record

def evaluate(artifact, tasks):
    jobs = [(artifact,t,r) for t in tasks for r in range(REPEATS)]
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        return list(pool.map(trajectory, jobs))

def summary(rows):
    return {"correct":sum(r["correct"] for r in rows), "attempts":len(rows),
        "bills":len({r["id"] for r in rows}),
        "tool_calls":sum(r.get("response",{}).get("action")=="tool" for r in rows),
        "api_errors":sum("api_error" in r for r in rows),
        "tokens":sum(r.get("tokens",0) for r in rows),
        "median_seconds":round(statistics.median(r.get("seconds",0) for r in rows),2)}

baseline_dev = evaluate(BASELINE, development)
print("Development baseline:", summary(baseline_dev))
for row in baseline_dev:
    if not row["correct"]:
        print(json.dumps(row))

Development baseline: {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 31518, 'median_seconds': 5.0}


## 3. Let the fixed improver choose
We use the same evidence for two proposals: an instruction-only change and an
open choice that can also write a tool or leave the agent unchanged. We give no
repair labels and never insist that a tool is better. The same fixed improver
instructions generate both proposals.

Generated tools must pass independent contract checks before any evaluation.
Candidates are retained only if they improve development accuracy, with no API
failures in the comparison. A tie retains the original. This conservative rule
makes “no demonstrated improvement” possible even when a proposal is sensible.

In [3]:
PROPOSAL_SCHEMA = {"type":"object", "properties":{
    "change":{"type":"string", "enum":["no_change", "instruction", "tool"]},
    "instruction":{"type":"string"}, "tool_code":{"type":"string"},
    "reason":{"type":"string"}}, "required":["change","instruction","tool_code","reason"],
    "additionalProperties":False}
IMPROVER = """You improve a bill-solving TASK agent. You cannot change yourself,
its model, evaluator, objective, or harness. Diagnose the supplied real development
trajectories, including correct answers. Do not invent failures. Choose no_change,
an instruction revision, or a reusable arithmetic tool. A tool is optional and
must be justified by evidence. The objective is exact cents under one model call
per bill. Keep task instructions under 150 words. For no_change, preserve the
original instruction and return empty tool_code. For instruction, return empty
tool_code. For tool, write exactly one Python function
split_bill(subtotal, tax, tip, diners, tip_on), taking dollar and percent strings,
an integer diners, and tip_on 'subtotal' or 'total'. Return the Decimal per-person
amount, with full intermediate precision and final ROUND_HALF_UP cents.
Provided: Decimal, str, int, ROUND_HALF_UP. Allowed: assignments, arithmetic,
conditional expressions, comparisons, Decimal.quantize. No imports, loops,
defaults, annotations, decorators, markdown fences, or other functions. Code must
be under 2500 characters. Explain the evidence for the choice in under 100 words.
"""

def propose(scope):
    proposal, resources = ask(IMPROVER + f"\nAllowed changes for this proposal: {scope}." +
        "\nOriginal agent: " + json.dumps(BASELINE) +
        "\nDevelopment trajectories: " + json.dumps(baseline_dev), PROPOSAL_SCHEMA)
    print(proposal["change"], "|", proposal["reason"])
    proposal["proposal_usage"] = resources
    if scope == "no_change or instruction" and proposal["change"] == "tool":
        proposal["validation_error"] = "Tool forbidden in instruction-only arm"
    elif proposal["change"] == "tool":
        try:
            for task in development:
                args = tuple(task[k] for k in PARAMS)
                result, tool = run_tool(proposal["tool_code"], "split_bill", args)
                assert result == expected(task), "Tool failed an independent contract check"
            proposal["tool"] = tool
        except Exception as error:
            proposal["validation_error"] = type(error).__name__ + ": " + str(error)
    elif proposal["tool_code"]:
        proposal["validation_error"] = "Unexpected code in a non-tool proposal"
    if proposal["change"] == "no_change":
        proposal["instruction"] = BASELINE["instruction"]
    return proposal

instruction_proposal = propose("no_change or instruction")
open_proposal = propose("no_change, instruction, or tool")
proposals = {"instruction_only":instruction_proposal, "open_choice":open_proposal}
artifacts, dev_rows = {"original":BASELINE}, {"original":baseline_dev}
for name, proposal in proposals.items():
    artifacts[name] = BASELINE if "validation_error" in proposal else proposal
    dev_rows[name] = evaluate(artifacts[name], development)
    print(name, summary(dev_rows[name]), "| validation:", proposal.get("validation_error","passed"))

# Decide retention before opening the held-out evaluation.
retained_name = "original"
if not any(summary(rows)["api_errors"] for rows in dev_rows.values()):
    for name in proposals:
        if sum(r["correct"] for r in dev_rows[name]) > sum(r["correct"] for r in dev_rows[retained_name]):
            retained_name = name
print("Retained:", retained_name)

no_change | All 32 development runs across ordinary, fractional, and rounding test cases achieved 100% accuracy and matched the reference amounts to exact cents in a single model call. Because no failures occurred and the agent reliably follows the intermediate precision and ROUND_HALF_UP rounding rules, no changes are justified.
no_change | Across all 48 runs spanning ordinary, fractional, and rounding test cases, the agent achieved 100% accuracy with zero calculation errors. Because no failures occurred in the development trajectories, introducing a tool or altering instructions is not justified by the evidence.
instruction_only {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 31449, 'median_seconds': 4.96} | validation: passed
open_choice {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 33817, 'median_seconds': 5.47} | validation: passed
Retained: original


## 4. Evaluate on unseen bills
We now evaluate all three frozen versions on the same held-out bills, including
rejected candidates as diagnostic comparisons. Held-out results never choose or
revise a version. The retained version was decided above. Error counts, paired
wins/losses, and results by bill group are more informative than one example.

In [4]:
held_out_rows = {name:evaluate(artifact, held_out) for name,artifact in artifacts.items()}
print("HELD-OUT RESULTS")
for name, rows in held_out_rows.items():
    print(name, summary(rows))
    for group in ("ordinary", "fractional", "rounding"):
        subset = [r for r in rows if r["group"] == group]
        print(" ", group, f"{sum(r['correct'] for r in subset)}/{len(subset)}")
    if name != "original":
        old = held_out_rows["original"]
        wins = sum(not a["correct"] and b["correct"] for a,b in zip(old,rows))
        losses = sum(a["correct"] and not b["correct"] for a,b in zip(old,rows))
        print("  paired wins/losses against original:", wins, losses)
print("Retained version:", retained_name)
print("Total calls:", len(usage), "| Reported tokens:", sum(r["tokens"] for r in usage))
# Everything needed to inspect or export trajectories remains in this session.
experiment = {"model":MODEL, "repeats":REPEATS, "retained":retained_name,
    "proposals":{k:{a:b for a,b in v.items() if a != "tool"} for k,v in proposals.items()},
    "development":dev_rows, "held_out":held_out_rows, "usage":usage}

HELD-OUT RESULTS
original {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 31050, 'median_seconds': 5.07}
  ordinary 16/16
  fractional 16/16
  rounding 16/16
instruction_only {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 32459, 'median_seconds': 5.22}
  ordinary 16/16
  fractional 16/16
  rounding 16/16
  paired wins/losses against original: 0 0
open_choice {'correct': 48, 'attempts': 48, 'bills': 24, 'tool_calls': 0, 'api_errors': 0, 'tokens': 33694, 'median_seconds': 5.11}
  ordinary 16/16
  fractional 16/16
  rounding 16/16
  paired wins/losses against original: 0 0
Retained version: original
Total calls: 290 | Reported tokens: 220109


## Read the result carefully
An accuracy gain on unseen bills supports this particular agent change. A tie
means we have not demonstrated an accuracy improvement. Tools may change token
cost or latency even on ties; proposal and development costs also belong in the
accounting. There are only 24 distinct held-out tasks, despite 48 attempts.

This synthetic experiment uses one model, one seed, one fixed improver, a known
formula, and a small action space. It does not show broad transfer or recursive
self-improvement. A stronger follow-up would repeat the whole experiment with
fresh seeds and independent proposals. First inspect the actual trajectories.

If both proposals choose `no_change`, the comparison arms are repeat evaluations
of the same policy. Their differences in token usage and latency are run
variability, not measured benefits of an agent change.


## Optional: inspect the recorded trajectories
Expand this JSON record to see actual responses and proposals. It contains no
credentials. Re-running this cell displays the current session's experiment.


In [5]:
from IPython.display import JSON, display
display(JSON(experiment, expanded=False))


Experiment record: 288 bill-solving trajectories and two improver proposals.